# 🩹 Notebook 3 — Sloppy Quorum, Hinted Handoff, and Read Repair

A **strict** quorum fails a write if fewer than `W` of the *preferred* replicas
for a key are up. Amazon's Dynamo paper introduces three techniques that keep
the system **available** during failures and **eventually consistent**
afterwards:

1. **Sloppy quorum** — if the preferred replicas are down, accept writes on
   *any* live node instead.
2. **Hinted handoff** — the stand-in node stores a note ("hint") and delivers
   the data to the real owner when it comes back.
3. **Read repair** — during a read, if replicas disagree, the coordinator
   pushes the freshest value back to the stale ones.

We'll implement all three in ~100 lines.


## 🏘️ Preferred replicas (consistent hashing, informally)

In Dynamo, each key `k` maps to a short ordered list of **preferred replicas**
— the "home" for that key. A strict quorum talks only to that list. We fake
this with a deterministic hash into the first N nodes of our cluster.


In [1]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import random, hashlib

@dataclass
class Node:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str, int]] = field(default_factory=dict)
    hints: List[Tuple[str, str, str, int]] = field(default_factory=list)
    # hints: (target_node_name, key, value, ts)

    def store(self, k, v, ts):
        if not self.up:
            return False
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:
            self.data[k] = (v, ts)
        return True

    def get(self, k):
        if not self.up:
            return None
        return self.data.get(k)


class DynamoLite:
    def __init__(self, num_nodes=6, N=3):
        self.nodes = [Node(f"n{i}") for i in range(num_nodes)]
        self.N = N                       # replication factor
        self._ts = 0

    def _preferred(self, key) -> List[Node]:
        '''Pick N preferred nodes deterministically from the key hash.'''
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        start = h % len(self.nodes)
        return [self.nodes[(start + i) % len(self.nodes)] for i in range(self.N)]

    def _fallback(self, exclude_names) -> Optional[Node]:
        for n in self.nodes:
            if n.up and n.name not in exclude_names:
                return n
        return None


## ✏️ Sloppy write with hinted handoff

If a preferred replica is down, the write is accepted by a **fallback** node
which stores both the data **and** a hint to forward it later.


In [2]:
def sloppy_write(self, key, value, W):
    self._ts += 1
    ts = self._ts
    preferred = self._preferred(key)
    preferred_names = {n.name for n in preferred}
    acks = 0
    used = set()  # set of node names
    for home in preferred:
        if home.up:
            home.store(key, value, ts)
            used.add(home.name)
            acks += 1
        else:
            # fallback must not be another preferred node (it will handle its own write)
            fb = self._fallback(exclude_names=used | preferred_names)
            if fb is None:
                continue
            fb.store(key, value, ts)
            fb.hints.append((home.name, key, value, ts))
            used.add(fb.name)
            acks += 1
    return (acks >= W), ts, acks, sorted(used)

DynamoLite.sloppy_write = sloppy_write


def deliver_hints(self):
    '''Replay hints for any node that came back up.'''
    by_name = {n.name: n for n in self.nodes}
    delivered = 0
    for holder in self.nodes:
        remaining = []
        for target_name, k, v, ts in holder.hints:
            target = by_name[target_name]
            if target.up and target.store(k, v, ts):
                delivered += 1
            else:
                remaining.append((target_name, k, v, ts))
        holder.hints = remaining
    return delivered

DynamoLite.deliver_hints = deliver_hints


## 👀 Sloppy write in action

In [3]:
db = DynamoLite(num_nodes=6, N=3)

# kill two of the three preferred replicas for key 'user:42'
pref = db._preferred("user:42")
print("preferred for user:42 →", [n.name for n in pref])
pref[0].up = False
pref[1].up = False

ok, ts, acks, used = db.sloppy_write("user:42", "Ada", W=3)
print(f"write ok={ok}, acks={acks}, landed on {used}")
print("hints held on stand-ins:",
      {n.name: n.hints for n in db.nodes if n.hints})

# the two preferred replicas come back
pref[0].up = True
pref[1].up = True
print("hints delivered:", db.deliver_hints())
print("final state on preferred replicas:")
for n in pref:
    print(f"  {n.name}.data = {n.data}")


preferred for user:42 → ['n0', 'n1', 'n2']
write ok=True, acks=3, landed on ['n2', 'n3', 'n4']
hints held on stand-ins: {'n3': [('n0', 'user:42', 'Ada', 1)], 'n4': [('n1', 'user:42', 'Ada', 1)]}
hints delivered: 2
final state on preferred replicas:
  n0.data = {'user:42': ('Ada', 1)}
  n1.data = {'user:42': ('Ada', 1)}
  n2.data = {'user:42': ('Ada', 1)}


The write survived even though **2 of the 3 preferred replicas were
down**. When they came back, the hint-holders quietly shipped the data over.

## 🔄 Read repair

Even with hints, replicas can drift. During a read, the coordinator can detect
stale copies and **push** the freshest value back.


In [4]:
def quorum_read_with_repair(self, key, R):
    responses = []
    for n in self.nodes:
        if not n.up:
            continue
        responses.append((n, n.get(key)))
        if len(responses) >= R:
            break
    with_data = [(n, v) for n, v in responses if v is not None]
    if not with_data:
        return None
    freshest_node, freshest = max(with_data, key=lambda x: x[1][1])
    # repair: push freshest value to any stale responder
    for n, v in responses:
        if v is None or v[1] < freshest[1]:
            n.store(key, freshest[0], freshest[1])
    return freshest_node.name, freshest

DynamoLite.quorum_read_with_repair = quorum_read_with_repair


# demo: create a stale replica manually
db = DynamoLite(num_nodes=5, N=3)
db.sloppy_write("k", "v1", W=3)
# drift: update only n0 to v2
db.nodes[0].data["k"] = ("v2", 99)
print("before repair:", {n.name: n.data.get("k") for n in db.nodes if "k" in n.data})
print("read →", db.quorum_read_with_repair("k", R=3))
print("after repair:", {n.name: n.data.get("k") for n in db.nodes if "k" in n.data})


before repair: {'n0': ('v2', 99), 'n3': ('v1', 1), 'n4': ('v1', 1)}
read → ('n0', ('v2', 99))
after repair: {'n0': ('v2', 99), 'n1': ('v2', 99), 'n2': ('v2', 99), 'n3': ('v1', 1), 'n4': ('v1', 1)}


## ⚖️ Conflict resolution: last-write-wins vs vector clocks

We've been using **last-write-wins (LWW)** — freshest timestamp beats older
ones. It's simple but loses data on concurrent writes (`Ada` and `Grace` both
edit the same key at exactly the same ms — one silently disappears).

Real Dynamo-style systems offer alternatives:

| Strategy             | Pros                       | Cons                                       |
|----------------------|----------------------------|--------------------------------------------|
| Last-write-wins      | simple, fast               | silent data loss on concurrent writes      |
| **Vector clocks**    | detects concurrent edits   | client must resolve (like git merge)       |
| **CRDTs**            | auto-merge, no conflicts   | limited data types (counters, sets, maps)  |

Riak exposed vector clocks to the client. DynamoDB chose LWW for simplicity.
Cassandra uses LWW by timestamp. Modern systems (Redis CRDT, Automerge,
Yjs) lean on CRDTs.


## 🧭 Recap

- **Strict quorum** (Notebook 1): strong consistency when `W + R > N`, but
  fails if too many preferred replicas are down.
- **Sloppy quorum + hinted handoff** (this notebook): trade a little
  consistency for **availability during failures**.
- **Read repair**: passively fix stale replicas on every read.
- **LWW vs vector clocks vs CRDTs**: how to resolve conflicts once you let
  them happen.

That's the whole picture of Dynamo-style quorum systems — the engine inside
DynamoDB, Cassandra, Riak, and Scylla.


## 🎯 Further exploration

- Read the original [Dynamo paper (2007)](https://www.allthingsdistributed.com/files/amazon-dynamo-sosp2007.pdf).
- Try running a real Cassandra cluster with `docker compose` and change
  `consistency_level` between `ONE`, `QUORUM`, `ALL` — watch latency and
  stale-read behaviour.
- Extend `DynamoLite` with **vector clocks** instead of LWW timestamps.
